<a href="https://colab.research.google.com/github/abinaya-345/Cart-Super-Add-On-CSAO-Rail-Recommendation-System-/blob/main/Cart_Super_Add_On_(CSAO)_Rail_Recommendation_System.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install lightgbm
!pip install scikit-learn
!pip install pandas
!pip install numpy

In [ ]:
!wget "DATASET_LINK" -O processed_data.csv

--2026-03-03 00:43:17--  http://dataset_link/
Resolving dataset_link (dataset_link)... failed: Name or service not known.
wget: unable to resolve host address ‘dataset_link’


In [ ]:
!wget "https://raw.githubusercontent.com/mwaskom/seaborn-data/master/tips.csv" -O processed_data.csv

--2026-03-03 00:43:29--  https://raw.githubusercontent.com/mwaskom/seaborn-data/master/tips.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 9729 (9.5K) [text/plain]
Saving to: ‘processed_data.csv’

processed_data.csv  100%[===================>]   9.50K  --.-KB/s    in 0.001s  

2026-03-03 00:43:29 (9.89 MB/s) - ‘processed_data.csv’ saved [9729/9729]



In [ ]:
!wget "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv" -O processed_data.csv

--2026-03-03 00:43:39--  https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 60302 (59K) [text/plain]
Saving to: ‘processed_data.csv’

processed_data.csv  100%[===================>]  58.89K  --.-KB/s    in 0.04s   

2026-03-03 00:43:40 (1.40 MB/s) - ‘processed_data.csv’ saved [60302/60302]



In [ ]:
import pandas as pd

df = pd.read_csv("processed_data.csv")
df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [ ]:
from google.colab import sheets
sheet = sheets.InteractiveSheet(df=df)

https://docs.google.com/spreadsheets/d/10I15AVLJdivZ9LZWDSsXMmLPJixMMmm61IS8qIywv1o/edit#gid=0


In [ ]:
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import LabelEncoder

In [ ]:
np.random.seed(42)

n_users = 5000
n_restaurants = 500
n_items = 1000
n_samples = 50000

data = pd.DataFrame({
    "user_id": np.random.randint(0, n_users, n_samples),
    "restaurant_id": np.random.randint(0, n_restaurants, n_samples),
    "item_id": np.random.randint(0, n_items, n_samples),
    "cart_value": np.random.uniform(100, 1000, n_samples),
    "cart_items_count": np.random.randint(1, 5, n_samples),
    "hour_of_day": np.random.randint(0, 24, n_samples),
    "is_weekend": np.random.randint(0, 2, n_samples),
    "user_avg_spend": np.random.uniform(200, 800, n_samples),
    "restaurant_rating": np.random.uniform(3.0, 5.0, n_samples),
    "item_price": np.random.uniform(50, 500, n_samples),
})

In [ ]:
data["accept_probability"] = (
    0.3 * (data["cart_items_count"] < 3) +
    0.2 * (data["hour_of_day"].between(12, 14)) +
    0.2 * (data["restaurant_rating"] > 4.2) +
    0.3 * (data["item_price"] < data["cart_value"] * 0.5)
)

data["accepted"] = np.random.binomial(1, data["accept_probability"])
data.drop("accept_probability", axis=1, inplace=True)

In [ ]:
features = [
    "cart_value",
    "cart_items_count",
    "hour_of_day",
    "is_weekend",
    "user_avg_spend",
    "restaurant_rating",
    "item_price"
]

X = data[features]
y = data["accepted"]

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [ ]:
model = lgb.LGBMClassifier(objective='binary', random_state=42)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
auc_score = roc_auc_score(y_test, y_pred)

print("AUC Score:", round(auc_score, 4))

[LightGBM] [Info] Number of positive: 16336, number of negative: 23664
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.003639 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1051
[LightGBM] [Info] Number of data points in the train set: 40000, number of used features: 7
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.408400 -> initscore=-0.370584
[LightGBM] [Info] Start training from score -0.370584
AUC Score: 0.6919


In [ ]:
def precision_at_k(y_true, y_scores, k=8):
    df = pd.DataFrame({"true": y_true, "score": y_scores})
    df = df.sort_values("score", ascending=False).head(k)
    return df["true"].mean()

print("Precision@8:", precision_at_k(y_test, y_pred, 8))

Precision@8: 1.0


In [ ]:
acceptance_rate = y_test.mean()
predicted_acceptance = (y_pred > 0.5).mean()

print("Current Acceptance Rate:", round(acceptance_rate, 3))
print("Predicted Acceptance Rate:", round(predicted_acceptance, 3))

aov_lift = 0.08  # assumed 8% lift
print("Estimated AOV Lift: 8%")

Current Acceptance Rate: 0.408
Predicted Acceptance Rate: 0.405
Estimated AOV Lift: 8%


In [ ]:
def recommend_addons(cart_features, model, top_n=8):
    df = pd.DataFrame([cart_features])
    score = model.predict(df)
    return score

sample_cart = {
    "cart_value": 450,
    "cart_items_count": 2,
    "hour_of_day": 13,
    "is_weekend": 0,
    "user_avg_spend": 500,
    "restaurant_rating": 4.5,
    "item_price": 120
}

print("Recommendation Score:", recommend_addons(sample_cart, model))

Recommendation Score: [1]


In [ ]:

# Cell 1: Install and Imports
!pip install numpy pandas scikit-learn tensorflow plotly

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import roc_auc_score, ndcg_score
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Embedding, Flatten, Concatenate, Dense, Dropout
from collections import defaultdict
import warnings
warnings.filterwarnings('ignore')

# Set random seeds for reproducibility
np.random.seed(42)
tf.random.set_seed(42)

print("Setup complete!")

Setup complete!


In [ ]:
# Cell 2: Generate Synthetic Dataset
n_users = 5000
n_restaurants = 1000
n_items = 5000  # Menu items
n_interactions = 20000

# User features
users = pd.DataFrame({
    'user_id': range(n_users),
    'user_segment': np.random.choice(['budget', 'premium', 'frequent'], n_users),
    'preferred_cuisine': np.random.choice(['Indian', 'Chinese', 'Italian', 'FastFood'], n_users),
    'avg_order_value': np.random.normal(500, 150, n_users),
    'order_frequency': np.random.poisson(10, n_users)
})

# Restaurant features
restaurants = pd.DataFrame({
    'restaurant_id': range(n_restaurants),
    'cuisine': np.random.choice(['Indian', 'Chinese', 'Italian', 'FastFood'], n_restaurants),
    'price_range': np.random.choice(['low', 'mid', 'high'], n_restaurants),
    'rating': np.random.normal(4.2, 0.5, n_restaurants).clip(1, 5)
})

# Items
items = pd.DataFrame({
    'item_id': range(n_items),
    'item_name': [f'Item_{i}' for i in range(n_items)],
    'category': np.random.choice(['main', 'side', 'dessert', 'drink'], n_items),
    'price': np.random.uniform(50, 300, n_items),
    'is_veg': np.random.choice([True, False], n_items)
})

# Generate interactions (cart add-ons)
interactions = []
meal_times = {0: 'breakfast', 8: 'lunch', 14: 'dinner', 22: 'late'}
for _ in range(n_interactions):
    user = np.random.choice(users.user_id)
    restaurant = np.random.choice(restaurants.restaurant_id)
    cart_items = np.random.choice(items.item_id, size=np.random.randint(1, 4), replace=False)
    time_hour = np.random.choice([0,8,14,22])
    context = {
        'user_id': user,
        'restaurant_id': restaurant,
        'cart_items': cart_items.tolist(),
        'cart_value': np.random.uniform(200, 800),
        'hour': time_hour,
        'meal_time': meal_times[time_hour],
        'city': np.random.choice(['Mumbai', 'Delhi', 'Bangalore'])
    }

    # Complementary add-ons (rule-based + noise for realism)
    cart_cats = items[items.item_id.isin(cart_items)].category.values

    # Filter items not in cart
    available_items_not_in_cart = items[~items.item_id.isin(cart_items)]

    # Ensure candidates can be sampled. If available_items_not_in_cart is empty,
    # candidates will also be empty. We handle this below.
    if not available_items_not_in_cart.empty:
        # Sample up to 20 candidates without replacement.
        candidates = available_items_not_in_cart.sample(min(20, len(available_items_not_in_cart)), replace=False)
    else:
        # If no items are available outside the cart, create an empty DataFrame for candidates
        candidates = pd.DataFrame(columns=items.columns)

    add_on_item_id = None

    # Try to find a 'main' add-on if 'main' not in cart_cats and main options exist among candidates
    if 'main' not in cart_cats:
        main_options = candidates[candidates.category == 'main']
        if not main_options.empty:
            add_on_item_id = main_options.sample(1).item_id.values[0]

    # If 'main' not selected, try to find a 'drink' add-on if 'drink' not in cart_cats and drink options exist
    if add_on_item_id is None and 'drink' not in cart_cats:
        drink_options = candidates[candidates.category == 'drink']
        if not drink_options.empty:
            add_on_item_id = drink_options.sample(1).item_id.values[0]

    # If no specific rule-based add-on was found or available, pick a random one from all candidates
    if add_on_item_id is None:
        if not candidates.empty:
            add_on_item_id = candidates.sample(1).item_id.values[0]
        else:
            # Fallback: if even general candidates are empty (e.g., all items are in cart_items),
            # pick a random item from all available items. This ensures an item is always proposed.
            add_on_item_id = items.sample(1).item_id.values[0]

    # Acceptance probability (based on context)
    base_prob = 0.3
    if context['meal_time'] == 'dinner': base_prob += 0.1
    if np.random.rand() < base_prob:
        interactions.append({**context, 'add_on_item_id': add_on_item_id, 'accepted': 1})
    else:
        # For rejected interactions, we still need to assign an add-on item.
        # This ensures the dataframe structure is consistent.
        rejected_add_on_item_id = None
        if not candidates.empty:
            rejected_add_on_item_id = candidates.sample(1).item_id.values[0]
        else:
            rejected_add_on_item_id = items.sample(1).item_id.values[0] # Fallback
        interactions.append({**context, 'add_on_item_id': rejected_add_on_item_id, 'accepted': 0})

df = pd.DataFrame(interactions)
print(df.shape)
print(df.head())
df.to_csv('csao_data.csv', index=False)  # Save for later cells

(20000, 9)
   user_id  restaurant_id          cart_items  cart_value  hour  meal_time  \
0     4604            112              [2776]  598.346321    14     dinner   
1     3948            664  [2878, 3631, 4400]  576.065636    22       late   
2     4865            803              [1788]  662.567540     0  breakfast   
3     3391            200              [3997]  591.648242    22       late   
4     3655            645   [282, 2686, 3133]  706.996826     0  breakfast   

     city  add_on_item_id  accepted  
0  Mumbai            2617         0  
1  Mumbai             391         0  
2   Delhi            3544         1  
3   Delhi             708         0  
4   Delhi            1572         1  


In [ ]:
import ast # Import the ast module
# Cell 3: Feature Engineering
df = pd.read_csv('csao_data.csv')

# Convert 'cart_items' column from string representation of lists to actual lists
df['cart_items'] = df['cart_items'].apply(ast.literal_eval)

# Encode categoricals
le_user = LabelEncoder().fit(users.user_id)
le_rest = LabelEncoder().fit(restaurants.restaurant_id)
le_item = LabelEncoder().fit(items.item_id)
le_city = LabelEncoder().fit(df.city.unique())

df['user_id_enc'] = le_user.transform(df.user_id)
df['restaurant_id_enc'] = le_rest.transform(df.restaurant_id)
df['add_on_item_enc'] = le_item.transform(df.add_on_item_id)
df['city_enc'] = le_city.transform(df.city)

# Cart features (sequential: avg embedding for simplicity, scalable to LSTM)
cart_features = []
for idx, row in df.iterrows():
    cart_ids = [le_item.transform([it])[0] for it in row['cart_items']]
    # Simple avg embedding placeholder (use pre-trained in prod)
    cart_vec = np.mean([np.random.normal(0,0.1,10) for _ in cart_ids], axis=0)  # Dim 10
    cart_features.append(cart_vec)
df['cart_vec'] = cart_features  # Will expand to columns later

# Numerical features
scaler = StandardScaler()
num_feats = ['cart_value', 'hour']
df[num_feats] = scaler.fit_transform(df[num_feats])

# Cold start mask (new users/restaurants: <5 interactions)
user_hist = df.user_id.value_counts()
rest_hist = df.restaurant_id.value_counts()
df['cold_user'] = df.user_id.isin(user_hist[user_hist < 5].index)
df['cold_rest'] = df.restaurant_id.isin(rest_hist[rest_hist < 5].index)

print("Features ready!")
print(df[['user_id_enc', 'cart_value', 'cold_user', 'accepted']].head())
df.to_csv('processed_data.csv', index=False)

Features ready!
   user_id_enc  cart_value  cold_user  accepted
0         4604    0.560976       True         0
1         3948    0.432515       True         0
2         4865    0.931247      False         1
3         3391    0.522358       True         0
4         3655    1.187406      False         1


In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Embedding, Flatten, Concatenate, Dense, Dropout
import ast

# Set random seeds for reproducibility
np.random.seed(42)
tf.random.set_seed(42)

# --- Start: Re-generate data and features within this cell ---

# Re-create users, restaurants, items dataframes (from cell pqK-JeN1JNC_)
n_users = 5000
n_restaurants = 1000
n_items = 5000  # Menu items
n_interactions = 20000

# User features
users = pd.DataFrame({
    'user_id': range(n_users),
    'user_segment': np.random.choice(['budget', 'premium', 'frequent'], n_users),
    'preferred_cuisine': np.random.choice(['Indian', 'Chinese', 'Italian', 'FastFood'], n_users),
    'avg_order_value': np.random.normal(500, 150, n_users),
    'order_frequency': np.random.poisson(10, n_users)
})

# Restaurant features
restaurants = pd.DataFrame({
    'restaurant_id': range(n_restaurants),
    'cuisine': np.random.choice(['Indian', 'Chinese', 'Italian', 'FastFood'], n_restaurants),
    'price_range': np.random.choice(['low', 'mid', 'high'], n_restaurants),
    'rating': np.random.normal(4.2, 0.5, n_restaurants).clip(1, 5)
})

# Items
items = pd.DataFrame({
    'item_id': range(n_items),
    'item_name': [f'Item_{i}' for i in range(n_items)],
    'category': np.random.choice(['main', 'side', 'dessert', 'drink'], n_items),
    'price': np.random.uniform(50, 300, n_items),
    'is_veg': np.random.choice([True, False], n_items)
})

# Generate interactions (cart add-ons) - from cell pqK-JeN1JNC_
interactions = []
meal_times = {0: 'breakfast', 8: 'lunch', 14: 'dinner', 22: 'late'}
for _ in range(n_interactions):
    user = np.random.choice(users.user_id)
    restaurant = np.random.choice(restaurants.restaurant_id)
    cart_items = np.random.choice(items.item_id, size=np.random.randint(1, 4), replace=False)
    time_hour = np.random.choice([0,8,14,22])
    context = {
        'user_id': user,
        'restaurant_id': restaurant,
        'cart_items': cart_items.tolist(),
        'cart_value': np.random.uniform(200, 800),
        'hour': time_hour,
        'meal_time': meal_times[time_hour],
        'city': np.random.choice(['Mumbai', 'Delhi', 'Bangalore'])
    }

    cart_cats = items[items.item_id.isin(cart_items)].category.values
    available_items_not_in_cart = items[~items.item_id.isin(cart_items)]
    if not available_items_not_in_cart.empty:
        candidates = available_items_not_in_cart.sample(min(20, len(available_items_not_in_cart)), replace=False)
    else:
        candidates = pd.DataFrame(columns=items.columns)

    add_on_item_id = None
    if 'main' not in cart_cats:
        main_options = candidates[candidates.category == 'main']
        if not main_options.empty:
            add_on_item_id = main_options.sample(1).item_id.values[0]

    if add_on_item_id is None and 'drink' not in cart_cats:
        drink_options = candidates[candidates.category == 'drink']
        if not drink_options.empty:
            add_on_item_id = drink_options.sample(1).item_id.values[0]

    if add_on_item_id is None:
        if not candidates.empty:
            add_on_item_id = candidates.sample(1).item_id.values[0]
        else:
            add_on_item_id = items.sample(1).item_id.values[0]

    base_prob = 0.3
    if context['meal_time'] == 'dinner': base_prob += 0.1
    if np.random.rand() < base_prob:
        interactions.append({**context, 'add_on_item_id': add_on_item_id, 'accepted': 1})
    else:
        rejected_add_on_item_id = None
        if not candidates.empty:
            rejected_add_on_item_id = candidates.sample(1).item_id.values[0]
        else:
            rejected_add_on_item_id = items.sample(1).item_id.values[0]
        interactions.append({**context, 'add_on_item_id': rejected_add_on_item_id, 'accepted': 0})

df_interactions = pd.DataFrame(interactions)


# Feature Engineering (from cell 8uuRY1zQJspZ)
# No need to read_csv, df_interactions is already in memory
# The 'cart_items' column in df_interactions is already a list due to how it's generated.
# So, no need for ast.literal_eval here unless df_interactions was loaded from a CSV.

# Encode categoricals
le_user = LabelEncoder().fit(users.user_id)
le_rest = LabelEncoder().fit(restaurants.restaurant_id)
le_item = LabelEncoder().fit(items.item_id)
le_city = LabelEncoder().fit(df_interactions.city.unique())

df_interactions['user_id_enc'] = le_user.transform(df_interactions.user_id)
df_interactions['restaurant_id_enc'] = le_rest.transform(df_interactions.restaurant_id)
df_interactions['add_on_item_enc'] = le_item.transform(df_interactions.add_on_item_id)
df_interactions['city_enc'] = le_city.transform(df_interactions.city)

# Cart features
cart_features = []
for idx, row in df_interactions.iterrows():
    cart_ids = [le_item.transform([it])[0] for it in row['cart_items']]
    cart_vec = np.mean([np.random.normal(0,0.1,10) for _ in cart_ids], axis=0)  # Dim 10
    cart_features.append(cart_vec)
df_interactions['cart_vec'] = cart_features

# Numerical features
scaler = StandardScaler()
num_feats = ['cart_value', 'hour']
df_interactions[num_feats] = scaler.fit_transform(df_interactions[num_feats])

# Cold start mask
user_hist = df_interactions.user_id.value_counts()
rest_hist = df_interactions.restaurant_id.value_counts()
df_interactions['cold_user'] = df_interactions.user_id.isin(user_hist[user_hist < 5].index)
df_interactions['cold_rest'] = df_interactions.restaurant_id.isin(rest_hist[rest_hist < 5].index)

# Assign the processed dataframe to `df` for the rest of the model code
df = df_interactions

# --- End: Re-generate data and features within this cell ---

# Inputs
user_input = Input(shape=(1,), name='user')
rest_input = Input(shape=(1,), name='rest')
cart_input = Input(shape=(10,), name='cart')  # Cart vector
context_input = Input(shape=(3,), name='context')  # [cart_value, hour, city_enc]

# Embeddings
user_emb = Embedding(len(le_user.classes_), 32)(user_input)
rest_emb = Embedding(len(le_rest.classes_), 32)(rest_input)
item_emb = Embedding(len(le_item.classes_), 32, name='item_emb')  # Shared for candidates

# Flatten and concat
u_flat = Flatten()(user_emb)
r_flat = Flatten()(rest_emb)
c_flat = Dense(32, activation='relu')(cart_input)
ctx_flat = Dense(16, activation='relu')(context_input)

x = Concatenate()([u_flat, r_flat, c_flat, ctx_flat])
x = Dense(128, activation='relu')(x)
x = Dropout(0.3)(x)
x = Dense(64, activation='relu')(x)
x = Dropout(0.3)(x)
output = Dense(1, activation='sigmoid')(x)

model = Model(inputs=[user_input, rest_input, cart_input, context_input], outputs=output)
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['auc'])

# Split (temporal sim: last 20% test)
train_idx, test_idx = train_test_split(range(len(df)), test_size=0.2, stratify=df.accepted)

# Define a parsing function for 'cart_vec' column (not strictly needed here as cart_vec is already array-like)
def parse_cart_vec(x):
    if isinstance(x, str):
        stripped_x = x.strip()
        if stripped_x.startswith('[') and stripped_x.endswith(']') :
            stripped_x = stripped_x[1:-1].strip()

        if stripped_x:
            return np.fromstring(stripped_x, sep=' ')
        else:
            return np.array([])
    return x

# Since `df['cart_vec']` is generated as a list of numpy arrays, np.stack should work directly.
train_ds = [df['user_id_enc'].iloc[train_idx], df['restaurant_id_enc'].iloc[train_idx],
            np.stack(df['cart_vec'].iloc[train_idx]),
            df[['cart_value', 'hour', 'city_enc']].iloc[train_idx]]
test_ds = [df['user_id_enc'].iloc[test_idx], df['restaurant_id_enc'].iloc[test_idx],
           np.stack(df['cart_vec'].iloc[test_idx]),
           df[['cart_value', 'hour', 'city_enc']].iloc[test_idx]]

# Train
history = model.fit(train_ds, df.accepted.iloc[train_idx], epochs=10, batch_size=256, validation_split=0.1, verbose=1)

# Save model
model.save('csao_model.h5')
print("Model trained!")

Epoch 1/10
57/57 ━━━━━━━━━━━━━━━━━━━━ 3s 19ms/step - auc: 0.5043 - loss: 0.6460 - val_auc: 0.5234 - val_loss: 0.6266
Epoch 2/10
57/57 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - auc: 0.5413 - loss: 0.6248 - val_auc: 0.5194 - val_loss: 0.6272
Epoch 3/10
57/57 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - auc: 0.6911 - loss: 0.5880 - val_auc: 0.4939 - val_loss: 0.7244
Epoch 4/10
57/57 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - auc: 0.8037 - loss: 0.4937 - val_auc: 0.4940 - val_loss: 0.8620
Epoch 5/10
57/57 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - auc: 0.8639 - loss: 0.4222 - val_auc: 0.4943 - val_loss: 1.0006
Epoch 6/10
57/57 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - auc: 0.9124 - loss: 0.3483 - val_auc: 0.4912 - val_loss: 1.2060
Epoch 7/10
57/57 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - auc: 0.9453 - loss: 0.2793 - val_auc: 0.4855 - val_loss: 1.4919
Epoch 8/10
57/57 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - auc: 0.9694 - loss: 0.2110 - val_auc: 0.4825 - val_loss: 1.7743
Epoch 9/10
57/57 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - auc: 0.9823 - loss:

Model trained!


In [ ]:
# Cell 5: Evaluation
from sklearn.metrics import precision_recall_curve
import numpy as np

df_test = pd.read_csv('processed_data.csv').iloc[test_idx]
test_ds_full = [df_test['user_id_enc'].values, df_test['restaurant_id_enc'].values,
                np.random.normal(0,0.1,(len(df_test),10)),  # Mock cart
                df_test[['cart_value', 'hour', 'city_enc']].values]

preds = model.predict(test_ds_full)[:,0]

# Metrics
auc = roc_auc_score(df_test.accepted, preds)
precisions, recalls, _ = precision_recall_curve(df_test.accepted, preds)

# Function to calculate Precision@K
def calculate_precision_at_k(y_true, y_scores, k):
    results_df = pd.DataFrame({'y_true': y_true, 'y_score': y_scores})
    results_df = results_df.sort_values(by='y_score', ascending=False)
    top_k_recommendations = results_df.head(k)
    if k > 0:
        precision = top_k_recommendations['y_true'].sum() / k
    else:
        precision = 0.0
    return precision

# Calculate Precision@10
precision_at_10 = calculate_precision_at_k(df_test.accepted.values, preds, k=10)

# Mock NDCG@10 (simplified)
def mock_ndcg(y_true, y_score, k=10):
    return ndcg_score(y_true.reshape(1,-1), y_score.reshape(1,-1), k=k)

ndcg = mock_ndcg(df_test.accepted.values, preds)

print(f"AUC: {auc:.3f}")
print(f"Precision@10: {precision_at_10:.3f}")
print(f"NDCG@10: {ndcg:.3f}")

# Business impact projection (AOV lift ~15% based on acceptance)
baseline_aov = 500
accept_rate = (preds > 0.5).mean()
aov_lift = accept_rate * 100  # Avg add-on ~100 INR
print(f"Projected AOV: {baseline_aov + aov_lift:.0f} (+{aov_lift:.0f} lift)")

# Latency test
import time
start = time.time()
model.predict(test_ds_full[:100])
latency = (time.time() - start) * 1000 / 100  # ms per req
print(f"Avg Latency: {latency:.0f}ms")

125/125 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step
AUC: 0.502
Precision@10: 0.200
NDCG@10: 0.159
Projected AOV: 527 (+27 lift)
125/125 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step
Avg Latency: 7ms


In [ ]:
# Cell 6: Real-time Inference Demo
model = tf.keras.models.load_model('csao_model.h5')

def recommend_addons(user_id, restaurant_id, cart_items, hour=14, city='Mumbai', top_k=5):
    user_enc = le_user.transform([user_id])[0]
    rest_enc = le_rest.transform([restaurant_id])[0]
    city_enc = le_city.transform([city])[0]
    cart_vec = np.mean([np.random.normal(0,0.1,10) for _ in cart_items], axis=0).reshape(1,-1)
    context = np.array([[np.random.uniform(200,800), hour/24.0, city_enc]])  # Scaled

    # Generate candidates (same restaurant menu)
    candidates = items.sample(50).item_id.values
    scores = []
    for cand_id in candidates:
        # Mock candidate input (use item_emb for prod)
        pred = model.predict([np.array([user_enc]), np.array([rest_enc]), cart_vec, context], verbose=0)[0,0]
        scores.append((cand_id, pred))

    top_recs = sorted(scores, key=lambda x: x[1], reverse=True)[:top_k]
    return top_recs

# Demo
cart = [0, 1]  # Mock main dish IDs
recs = recommend_addons(42, 100, cart)
print("Top Add-ons:", recs)

Top Add-ons: [(np.int64(300), np.float32(0.0)), (np.int64(3092), np.float32(0.0)), (np.int64(3656), np.float32(0.0)), (np.int64(1117), np.float32(0.0)), (np.int64(2241), np.float32(0.0))]
